# Visual verification gallery

This notebook regenerates the documentation figures and displays their numerical and analytical references. Each script runs the actual finite element calculations and checks the properties stated in its case page. Performance figures read existing measurements and do not rerun timed experiments.

The reference panels are evaluated from analytical formulas. They are not digitized paper figures. Numerical and exact field panels share color limits; error panels have separate scales. See the case pages for discretizations, quadrature errors and measured norms.

In [ ]:
from pathlib import Path
import subprocess
import sys
from IPython.display import Image, display

root = Path.cwd()
if not (root / "examples").is_dir():
    root = root.parent
assert (root / "examples").is_dir(), "Run from the repository or notebooks directory"

def render(script):
    """Regenerate checked figures with the current notebook Python environment."""
    completed = subprocess.run(
        [sys.executable, str(root / "examples" / script)],
        cwd=root, check=True, text=True, capture_output=True,
    )
    print(f"{script}: completed")
    if completed.stderr:
        print(completed.stderr)

def show_figures(folder):
    """Display every exported PNG in a case family at readable notebook width."""
    figures = sorted((root / "docs" / "figures" / folder).glob("*.png"))
    assert figures, f"No figures generated for {folder}"
    for path in figures:
        print(path.stem)
        display(Image(filename=str(path), width=1200))

## Darcy: pressure, physical flux and conservation

The cosine pressure case compares primal P1 and mixed RT0/P0 fields. The layered case should show a pressure-slope change at the material interface and a constant physical flux. A P0 pressure field has projection error even when its RT0 flux is exact. Equilibration should restore fine-cell balance; the raw primal gradient need not have that property. Detailed equations and results: [Darcy case page](../docs/cases/darcy.md).

In [ ]:
render('plot_darcy_cases.py')
show_figures('darcy')

## Stokes, Brinkman and Oseen

The polynomial streamfunction gives an exactly divergence-free analytical velocity and a known pressure. Numerical weak incompressibility does not imply pointwise zero divergence; the figures show it explicitly. Brinkman and Oseen forces are derived to retain the prescribed analytical fields, so these are consistency comparisons rather than a fixed-force drag sweep. See [flow cases](../docs/cases/flow.md).

In [ ]:
render('plot_flow_cases.py')
show_figures('flow')

## Transport, heat and elasticity

The RAD field should follow the prescribed sine profile. The heat eigenmode should keep its shape and decay; backward Euler retains a larger amplitude than the continuum reference for a finite time step. The displacement case should move the interior in direction (1, 1/2) with approximately second-order L2 convergence. The deformed mesh uses a display factor 0.2, and weak boundary conditions can leave small nodal deviations. See [scalar cases](../docs/cases/scalars.md) and [elasticity](../docs/cases/elasticity.md).

In [ ]:
render('plot_scalar_elasticity.py')
show_figures('scalar-elasticity')

## Measured CPU and GPU performance

The reference speedup is one. These recorded small workloads did not benefit from multiprocessing or GPU solves. Error bars show the minimum and maximum of three runs, not confidence intervals. No new timing run is executed by this notebook. See [performance](../docs/performance.md).

In [ ]:
render('plot_performance.py')
show_figures('performance')